# BERT (Devlin et al., 2018) — Implementation / 구현

**Paper**: Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding." NAACL-HLT 2019. arXiv:1810.04805.

**한국어**
이 노트북은 **mini-BERT**를 PyTorch로 직접 구현합니다. 원 논문의 BERT-Base는 12-layer × 768-hidden × 12-head × 110M params이지만, 여기서는 노트북에서 빠르게 학습 가능한 작은 버전(2-layer × 64-hidden × 4-head)을 만들어 핵심 메커니즘을 시연합니다. 시연 항목: (1) token + segment + position 임베딩 합산, (2) Transformer encoder block, (3) Masked Language Modeling head, (4) 80/10/10 마스킹 절차, (5) 작은 합성 corpus에서의 MLM 학습 곡선, (6) self-attention weight 시각화.

**English**
This notebook implements a **mini-BERT** from scratch in PyTorch. The paper's BERT-Base is 12L × 768H × 12A × 110M params; here we use a tiny version (2L × 64H × 4A) trainable in seconds inside a notebook so we can showcase the core mechanisms. Demonstrations: (1) summed token + segment + position embeddings, (2) a Transformer encoder block, (3) the Masked LM head, (4) the 80/10/10 masking procedure, (5) MLM training curve on a tiny synthetic corpus, (6) self-attention weight visualization.

**Kernel**: `study-with-ai`

In [ ]:
import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Part 1: Tiny Synthetic Corpus / 작은 합성 코퍼스

**한국어**
BERT 사전학습은 BooksCorpus(800M words) + Wikipedia(2,500M words)에서 1M step 학습됩니다. 우리는 패턴이 단순한 합성 문장 100개를 만듭니다 — 동물 + 동작 + 장소 형태의 SVO 문장들. 모델이 마스킹된 단어를 주변 문맥(어떤 동물인가 / 어떤 장소인가)으로부터 추론하는 능력을 검증할 수 있도록 충분한 통계적 의존성을 부여합니다.

**English**
BERT is pre-trained on BooksCorpus (800M words) + Wikipedia (2,500M words) for 1M steps. We instead build 100 synthetic sentences with a simple SVO pattern — animal + action + location — designed so a model can recover masked tokens from neighboring context (which animal? which location?).

In [ ]:
# Build a tiny vocabulary and synthetic corpus.

ANIMALS = ['cat', 'dog', 'bird', 'fish', 'horse']
ACTIONS = ['eats', 'sees', 'chases', 'likes', 'finds']
OBJECTS = ['food', 'water', 'toy', 'friend', 'home']
PLACES = ['park', 'house', 'river', 'forest', 'garden']
CONNECTORS = ['in', 'at', 'near', 'by']

SPECIAL = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']

WORDS = ANIMALS + ACTIONS + OBJECTS + PLACES + CONNECTORS
VOCAB = SPECIAL + sorted(set(WORDS))
TOK2ID = {tok: i for i, tok in enumerate(VOCAB)}
ID2TOK = {i: tok for tok, i in TOK2ID.items()}
VOCAB_SIZE = len(VOCAB)

PAD_ID = TOK2ID['[PAD]']
CLS_ID = TOK2ID['[CLS]']
SEP_ID = TOK2ID['[SEP]']
MASK_ID = TOK2ID['[MASK]']


def random_sentence(rng: random.Random) -> List[str]:
    """Generates a random SVO+location sentence as a list of tokens."""
    return [
        rng.choice(ANIMALS),
        rng.choice(ACTIONS),
        rng.choice(OBJECTS),
        rng.choice(CONNECTORS),
        rng.choice(PLACES),
    ]


rng = random.Random(SEED)
CORPUS = [random_sentence(rng) for _ in range(100)]

print(f'Vocab size: {VOCAB_SIZE}')
print(f'Corpus size: {len(CORPUS)} sentences')
print('First 3 sentences:')
for s in CORPUS[:3]:
    print('  ', ' '.join(s))

## Part 2: Mini-BERT Architecture / Mini-BERT 구조

**한국어**
BERT의 핵심 구성요소를 그대로 옮깁니다.

1. **Embedding layer**: $E_i = E^{tok}_i + E^{seg}_i + E^{pos}_i$ (token + segment + position 임베딩의 합) + LayerNorm + dropout.
2. **Transformer encoder block**: multi-head self-attention → residual + LayerNorm → position-wise FFN(GELU, hidden $4H$) → residual + LayerNorm.
3. **MLM head**: 마스킹된 위치의 final hidden state를 vocabulary 위 softmax로 보냄. token embedding과 weight tying.

**English**
We faithfully port BERT's core components.

1. **Embedding layer**: $E_i = E^{tok}_i + E^{seg}_i + E^{pos}_i$ (sum of token + segment + position embeddings) + LayerNorm + dropout.
2. **Transformer encoder block**: multi-head self-attention → residual + LayerNorm → position-wise FFN (GELU, hidden $4H$) → residual + LayerNorm.
3. **MLM head**: feeds final hidden states at masked positions through a softmax over vocabulary; weight-tied with the token embedding.

In [ ]:
class BERTEmbedding(nn.Module):
    """Token + Segment + Position embedding sum, with LayerNorm and dropout."""

    def __init__(self, vocab_size: int, hidden_size: int, max_len: int = 64,
                 num_segments: int = 2, dropout: float = 0.1) -> None:
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, hidden_size, padding_idx=PAD_ID)
        self.seg_emb = nn.Embedding(num_segments, hidden_size)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids: torch.Tensor, segment_ids: torch.Tensor) -> torch.Tensor:
        """Args:
            token_ids: LongTensor of shape (B, T).
            segment_ids: LongTensor of shape (B, T) with values in {0, 1}.

        Returns:
            Tensor of shape (B, T, H).
        """
        b, t = token_ids.shape
        positions = torch.arange(t, device=token_ids.device).unsqueeze(0).expand(b, t)
        emb = self.tok_emb(token_ids) + self.seg_emb(segment_ids) + self.pos_emb(positions)
        return self.dropout(self.layer_norm(emb))


class MultiHeadSelfAttention(nn.Module):
    """Standard multi-head self-attention with optional padding mask."""

    def __init__(self, hidden_size: int, num_heads: int, dropout: float = 0.1) -> None:
        super().__init__()
        assert hidden_size % num_heads == 0
        self.h = num_heads
        self.dk = hidden_size // num_heads
        self.qkv = nn.Linear(hidden_size, hidden_size * 3)
        self.out = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Args:
            x: (B, T, H).
            attn_mask: (B, T) bool, True for valid (non-pad) tokens.

        Returns:
            (out, attn): out is (B, T, H); attn is (B, h, T, T) for inspection.
        """
        b, t, _ = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each (B, h, T, dk)

        scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.dk)  # (B, h, T, T)
        # Mask pads in keys: positions where attn_mask is False get -inf in their column.
        key_mask = attn_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)
        scores = scores.masked_fill(~key_mask, float('-inf'))
        attn = scores.softmax(dim=-1)
        attn = self.dropout(attn)
        ctx = (attn @ v).transpose(1, 2).contiguous().view(b, t, -1)
        return self.out(ctx), attn


class TransformerBlock(nn.Module):
    """BERT-style Transformer encoder block: MHA + FFN with residual + LayerNorm."""

    def __init__(self, hidden_size: int, num_heads: int, ffn_size: int,
                 dropout: float = 0.1) -> None:
        super().__init__()
        self.attn = MultiHeadSelfAttention(hidden_size, num_heads, dropout)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        a, attn = self.attn(x, attn_mask)
        x = self.ln1(x + self.dropout(a))
        f = self.ffn(x)
        x = self.ln2(x + self.dropout(f))
        return x, attn


class MiniBERT(nn.Module):
    """A small BERT model for pedagogical MLM training."""

    def __init__(self, vocab_size: int, hidden_size: int = 64, num_layers: int = 2,
                 num_heads: int = 4, max_len: int = 32, dropout: float = 0.1) -> None:
        super().__init__()
        self.embedding = BERTEmbedding(vocab_size, hidden_size, max_len=max_len, dropout=dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_size, num_heads, ffn_size=4 * hidden_size, dropout=dropout)
            for _ in range(num_layers)
        ])
        # MLM head: project to hidden, GELU, LN, then linear to vocab.
        self.mlm_dense = nn.Linear(hidden_size, hidden_size)
        self.mlm_ln = nn.LayerNorm(hidden_size)
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, token_ids: torch.Tensor, segment_ids: torch.Tensor,
                attn_mask: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """Returns MLM logits (B, T, V) and a list of per-layer attention tensors."""
        h = self.embedding(token_ids, segment_ids)
        attns = []
        for block in self.blocks:
            h, attn = block(h, attn_mask)
            attns.append(attn)
        x = F.gelu(self.mlm_dense(h))
        x = self.mlm_ln(x)
        # Weight-tied output projection with token embedding (BERT trick).
        logits = x @ self.embedding.tok_emb.weight.T + self.mlm_bias
        return logits, attns


model = MiniBERT(VOCAB_SIZE, hidden_size=64, num_layers=2, num_heads=4, max_len=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {n_params:,}')

## Part 3: 80/10/10 Masking Procedure / 80/10/10 마스킹 절차

**한국어**
BERT의 마스킹 전략을 그대로 구현합니다. 입력 토큰의 15%를 무작위로 선택한 뒤:
- 80%는 `[MASK]`로 대체
- 10%는 무작위 vocabulary 토큰으로 대체
- 10%는 그대로 유지 (그러나 그 위치도 예측 대상)

특수 토큰(`[CLS]`, `[SEP]`, `[PAD]`)은 마스킹에서 제외합니다. 손실은 마스킹된 위치의 cross-entropy만 계산.

**English**
We faithfully port BERT's masking strategy. We sample 15% of input tokens, then:
- 80% are replaced with `[MASK]`
- 10% are replaced with a random vocabulary token
- 10% are left unchanged (but still predicted)

Special tokens (`[CLS]`, `[SEP]`, `[PAD]`) are excluded from masking. The loss is cross-entropy over masked positions only.

In [ ]:
MAX_LEN = 16  # plenty for our tiny SVO sentences with [CLS] [SEP] padding
MASK_PROB = 0.15

SPECIAL_IDS = {PAD_ID, CLS_ID, SEP_ID, MASK_ID, TOK2ID['[UNK]']}


def encode_sentence(tokens: List[str]) -> Tuple[List[int], List[int]]:
    """Wraps a token list into [CLS] ... [SEP] with segment ids and pads to MAX_LEN."""
    ids = [CLS_ID] + [TOK2ID.get(t, TOK2ID['[UNK]']) for t in tokens] + [SEP_ID]
    seg = [0] * len(ids)
    while len(ids) < MAX_LEN:
        ids.append(PAD_ID)
        seg.append(0)
    return ids[:MAX_LEN], seg[:MAX_LEN]


def apply_mlm_mask(token_ids: torch.Tensor, rng: np.random.Generator,
                   p_mask: float = MASK_PROB) -> Tuple[torch.Tensor, torch.Tensor]:
    """Applies BERT's 80/10/10 masking; returns (masked_inputs, labels).

    Labels are set to -100 at non-masked positions so cross_entropy ignores them.
    """
    inputs = token_ids.clone()
    labels = torch.full_like(token_ids, -100)
    b, t = token_ids.shape
    for i in range(b):
        for j in range(t):
            tid = int(token_ids[i, j].item())
            if tid in SPECIAL_IDS:
                continue
            if rng.random() < p_mask:
                labels[i, j] = tid  # remember the original
                r = rng.random()
                if r < 0.8:
                    inputs[i, j] = MASK_ID
                elif r < 0.9:
                    # random non-special token
                    while True:
                        cand = int(rng.integers(0, VOCAB_SIZE))
                        if cand not in SPECIAL_IDS:
                            break
                    inputs[i, j] = cand
                # else: keep unchanged (10%)
    return inputs, labels


# Build the full encoded dataset.
encoded = [encode_sentence(s) for s in CORPUS]
all_ids = torch.tensor([e[0] for e in encoded], dtype=torch.long)
all_seg = torch.tensor([e[1] for e in encoded], dtype=torch.long)
all_mask = (all_ids != PAD_ID)
print(f'Encoded shape: {tuple(all_ids.shape)}')

# Demo one masking.
demo_rng = np.random.default_rng(0)
demo_in, demo_lbl = apply_mlm_mask(all_ids[:1], demo_rng)
print('\nOriginal :', ' '.join(ID2TOK[int(t)] for t in all_ids[0].tolist()))
print('Masked   :', ' '.join(ID2TOK[int(t)] for t in demo_in[0].tolist()))
print('Labels   :', ' '.join(ID2TOK[int(t)] if int(t) != -100 else '-' for t in demo_lbl[0].tolist()))

## Part 4: Training the Mini-BERT with MLM / Mini-BERT를 MLM으로 학습

**한국어**
AdamW로 작은 학습률(1e-3)을 사용해 200 epoch 학습합니다. epoch마다 80/10/10 마스킹을 새로 샘플링하여 동일 문장에서 다양한 마스킹 패턴을 학습하게 합니다. 손실 곡선이 감소하면 모델이 문맥으로부터 마스킹된 토큰을 예측하는 데 성공하고 있음을 의미합니다.

**English**
We train for 200 epochs with AdamW (lr=1e-3). The 80/10/10 mask is **re-sampled every epoch**, so the same sentence appears with many different masking patterns. A decreasing loss curve confirms the model is learning to recover masked tokens from context.

In [ ]:
EPOCHS = 200
BATCH_SIZE = 16
LR = 1e-3

optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_history: List[float] = []
rng = np.random.default_rng(SEED)

all_ids_dev = all_ids.to(DEVICE)
all_seg_dev = all_seg.to(DEVICE)
all_mask_dev = all_mask.to(DEVICE)

for epoch in range(EPOCHS):
    # Re-sample masks every epoch.
    inputs, labels = apply_mlm_mask(all_ids, rng)
    inputs = inputs.to(DEVICE)
    labels = labels.to(DEVICE)

    # Mini-batch shuffle.
    perm = torch.randperm(inputs.size(0))
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, inputs.size(0), BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        bi = inputs[idx]
        bs = all_seg_dev[idx]
        bm = all_mask_dev[idx]
        bl = labels[idx]
        logits, _ = model(bi, bs, bm)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), bl.view(-1), ignore_index=-100)
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        epoch_loss += float(loss.item())
        n_batches += 1
    loss_history.append(epoch_loss / max(n_batches, 1))
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch + 1:3d}/{EPOCHS} | MLM loss = {loss_history[-1]:.4f}')

fig, ax = plt.subplots()
ax.plot(loss_history, color='#1f77b4')
ax.set_xlabel('Epoch')
ax.set_ylabel('MLM loss / MLM 손실')
ax.set_title('Mini-BERT MLM training curve / Mini-BERT MLM 학습 곡선')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5: Qualitative Predictions / 정성적 예측

**한국어**
학습된 mini-BERT가 임의 문장에서 한 토큰을 가렸을 때 어떤 후보를 제시하는지 확인합니다. SVO 문장 패턴을 학습했다면, 동물 자리에는 동물이, 장소 자리에는 장소가 더 높은 확률로 예측되어야 합니다.

**English**
We probe the trained mini-BERT by masking one token of a sentence and inspecting the top-5 vocabulary candidates. If it has learned the SVO pattern, the animal slot should produce animal candidates and the place slot should produce place candidates.

In [ ]:
@torch.no_grad()
def predict_topk(sentence: List[str], mask_pos: int, k: int = 5) -> List[Tuple[str, float]]:
    """Returns the top-k predictions (token, prob) at `mask_pos` (1-indexed inside the sentence)."""
    model.eval()
    ids, seg = encode_sentence(sentence)
    masked_ids = ids.copy()
    abs_pos = mask_pos  # because we add [CLS] at index 0, and `mask_pos` here is index in `ids`.
    masked_ids[abs_pos] = MASK_ID
    t_ids = torch.tensor([masked_ids], device=DEVICE)
    t_seg = torch.tensor([seg], device=DEVICE)
    t_mask = (t_ids != PAD_ID)
    logits, _ = model(t_ids, t_seg, t_mask)
    probs = logits[0, abs_pos].softmax(dim=-1)
    top = torch.topk(probs, k)
    return [(ID2TOK[int(i)], float(p)) for i, p in zip(top.indices, top.values)]


test_sentence = ['cat', 'eats', 'food', 'in', 'park']
encoded_ids, _ = encode_sentence(test_sentence)
print('Sentence layout (with [CLS]/[SEP]):',
      ' '.join(f'{i}:{ID2TOK[t]}' for i, t in enumerate(encoded_ids[:len(test_sentence) + 2])))
print()

for slot_name, pos in [('animal (pos 1)', 1), ('action (pos 2)', 2),
                       ('object (pos 3)', 3), ('place (pos 5)', 5)]:
    preds = predict_topk(test_sentence, pos, k=5)
    print(f'Mask {slot_name}: top-5 = {preds}')

## Part 6: Visualizing Self-Attention / Self-Attention 시각화

**한국어**
BERT의 강점은 모든 layer의 self-attention이 양방향이라는 점입니다. 한 sample에서 마지막 layer의 head 0 attention 행렬을 그려, 어떤 토큰이 어떤 토큰을 참조하는지 시각적으로 확인합니다. 행 = query, 열 = key, 셀 값 = attention weight.

**English**
BERT's strength is that **every layer's** self-attention is bidirectional. We plot the attention matrix of head 0 in the last layer for one sample. Rows = query positions, columns = key positions, cell value = attention weight.

In [ ]:
@torch.no_grad()
def attention_for(sentence: List[str]) -> Tuple[np.ndarray, List[str]]:
    """Returns last-layer head-0 attention matrix and the token labels (no padding)."""
    model.eval()
    ids, seg = encode_sentence(sentence)
    valid_len = sum(1 for t in ids if t != PAD_ID)
    t_ids = torch.tensor([ids], device=DEVICE)
    t_seg = torch.tensor([seg], device=DEVICE)
    t_mask = (t_ids != PAD_ID)
    _, attns = model(t_ids, t_seg, t_mask)
    last = attns[-1][0, 0].cpu().numpy()  # (T, T)
    labels = [ID2TOK[t] for t in ids[:valid_len]]
    return last[:valid_len, :valid_len], labels


att, labels = attention_for(test_sentence)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(att, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_xlabel('Key position / 키 위치')
ax.set_ylabel('Query position / 쿼리 위치')
ax.set_title('Last-layer head-0 attention (mini-BERT) / 마지막 layer head 0 attention')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print('\nObservation / 관찰: each row sums to 1 (softmax over keys); the bidirectional')
print('character is visible because both columns to the left AND to the right of the diagonal')
print('can carry significant weight — unlike a causal LM where the upper triangle would be zero.')

## Summary / 요약

**한국어**
이 노트북은 BERT의 핵심 메커니즘을 처음부터 구현하여 다음을 보였습니다.

- **양방향 self-attention** 만으로도 마스킹된 토큰을 좌우 문맥에서 복원할 수 있음.
- **80/10/10 마스킹**은 사전학습/fine-tuning mismatch를 줄이는 단순하지만 효과적인 트릭임.
- **Token + segment + position 임베딩 합산**이라는 단순한 입력 표현이 sentence-pair 과제까지 통합 처리 가능.

원 논문(BERT-Base 110M params, BERT-Large 340M params)과 비교하면 우리 mini-BERT는 수만 파라미터에 불과하지만, 동일한 객체와 동일한 학습 신호를 사용합니다.

**English**
This notebook implemented BERT's core mechanisms from scratch and showed:

- **Bidirectional self-attention** alone can recover masked tokens using both-side context.
- The **80/10/10 masking** trick is a simple but effective fix for the pre-training/fine-tuning mismatch.
- A simple **token + segment + position embedding sum** unifies single-sentence and sentence-pair tasks.

Compared to the paper (BERT-Base 110M params, BERT-Large 340M params), our mini-BERT has only tens of thousands of parameters, but uses the same architecture and the same training signal.

| Concept / 개념 | This notebook / 이 노트북 | Paper (BERT-Base) / 논문 (BERT-Base) |
|---|---|---|
| Layers (L) | 2 | 12 |
| Hidden size (H) | 64 | 768 |
| Attention heads (A) | 4 | 12 |
| FFN inner / FFN 내부 | 256 (=4H) | 3072 (=4H) |
| Vocab / 어휘 크기 | ~30 | 30,000 (WordPiece) |
| Corpus / 말뭉치 | 100 synthetic SVO sentences / 100개 합성 SVO 문장 | BooksCorpus + Wikipedia (~3.3B words) |
| Pre-training tasks / 사전학습 과제 | MLM only / MLM만 | MLM + NSP |
| Steps | 200 epochs × 7 batches | 1,000,000 steps × 256 batch |
| Hardware | Notebook CPU/GPU / 노트북 CPU/GPU | 16 Cloud TPUs × 4 days |

**Next steps / 다음 단계**: Add NSP head, scale H and L, swap the synthetic corpus for a real text dataset, and fine-tune on a downstream classification task to fully reproduce the paper's recipe.